# Week 4: Temporal-Difference Learning

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab explores Temporal-Difference (TD) learning, which overcomes limitations present in the methods we've learned so far. Like Dynamic Programming (DP), TD is based on bootstrapping; that is, it uses estimates rather than outcomes to learn optimal policy. However, TD does not require a model of environment dynamics to derive estimates; it learns from experience what rewards are obtained from state-action pairs. That experiential approach is shared with Monte Carlo (MC) methods, but where MC learns from complete episodes and final returns, TD dynamically updates estimates after each step using the observed reward and the estimated value of the next state. This means TD can work where other methods are impractical: it doesn't need a model like DP, and it doesn't require complete episodes like MC.

As in prior methods, exploration versus exploitation is still the central trade-off. Greedy optimization risks insufficient exploration. DP can rely on a model to steer learning; when learning is based on experience, an exploration coefficient has to be used instead.

In Chapter 6 of "Reinforcement Learning: An introduction", Sutton & Barto describe two algorithms for addressing this problem (p. 100): SARSA, an "on-policy" approach which "attempt[s] to evaluate or improve the policy that is used to make decisions," and Q-learning, an "off-policy" method which "evaluate\[s] or improve\[s] a policy different from that used to generate the data."

The key difference between SARSA and Q-learning is in the way that they learn (i.e. update Q-values). SARSA learns based on the next action it takes. Q-learning learns based on the max value of the next possible actions. Where SARSA asks, "what happens if I sometimes act randomly", Q-learning asks, "what happens if I always act optimally."

This lab uses the Cliff Walking Gymnasium environment to understand the differences between SARSA and Q-learning, and the effects that they produce. The Cliff Walking environment is a 4x12 grid, with 37 reachable observation states, 10 "over the cliff" states, and one goal state.

From each cell, the agent can take one of four actions: up, down, right, and left. The agent incurs a penalty of -1 for each step it takes in the 37 reachable "solid ground" cells. Stepping into one of the 10 "over the cliff" states incurs a penalty of -100 and resets the agent back to start. Reaching the goal state terminates the episode.

We will implement a SARSA agent and a Q-learning agent. Both agents will sometimes fall off the cliff and incur the -100 penalty. The main difference is that SARSA will merge those penalties into the Q-values; Q-learning's max operator will preserve the more optimistic estimates. By this mechanism, we can expect the SARSA agent to learn to avoid the row next to the cliff, but we can expect the Q-learning agent to learn that the row next to the cliff is the shortest path to the goal, even if it is the one with the greatest risk.

## Section 2: Deliverables

In [ ]:
'''
Weight: 35 points (35%)

This section combines your implementation summary, results, and analysis.
'''

### GitHub Repository

GitHub Repository found here:  https://github.com/craig-rudman/MSDS684/tree/main/W4

In [ ]:
'''GitHub Repository URL (Required)
Place prominently at the top of this section:

GitHub Repository: https://github.com/username/repo/tree/main/lab#

Note: Your repository contains complete code, all experiments, and additional visualizations. 
This section highlights key findings with deep analysis.

What’s in GitHub Repository
All code (runnable, reproducible)
All generated visualizations
README with setup instructions
Additional experiments beyond what’s in PDF
requirements.txt or environment.yml
'''


### Implementation Summary

In [ ]:
'''Implementation Summary (100-150 words)

Brief prose description:

What you implemented (algorithms, environments)
Experimental setup (e.g., “1000 episodes, 30 random seeds, ε=0.1”)
Key hyperparameters chosen
NOT detailed pseudocode or line-by-line methods
'''


#### Key Results and Analysis

In [ ]:
'''Key Results & Analysis (400-600 words + 2-4 visualizations)
⚠️ CRITICAL RULES:

NO raw code listings in PDF
NO console output dumps
Code lives in GitHub; analysis lives here
Include 2-4 key visualizations:

Embedded directly in PDF (PNG/JPG format)
Each must have detailed interpretive caption
Choose your most important/insightful results
Captions must be interpretive, not just descriptive:

❌ Bad caption: “Figure 1 shows a line going up”

✅ Good caption: “Figure 1: UCB’s regret curve flattens after 500 steps because the
uncertainty bonus decreases as actions are sampled more, focusing exploration on 
genuinely uncertain actions (S&B Ch. 2.7). 
Final regret is 15% lower than ε-greedy (ε=0.1), demonstrating more efficient exploration.”

Discussion must address:

What do results show about algorithm behavior?
How do they relate to theory from Sutton & Barto? (cite chapters/sections)
What didn’t work as expected? Why?
How did hyperparameters affect performance?
What does this teach you about the RL concept?
Focus: Interpretation and understanding, not exhaustive description of every result.
'''

## Section 3: AI Use Reflection

In [ ]:
'''
(250-350 words)
Weight: 35 points (35%)

This is the most essential section - it captures the learning process.
'''

### 3.1 Initial Interaction

I asked Claude to help me plan a TDD implementation of SARSA and Q-learning on Gymnasium's CliffWalking-v0. My initial prompt was the full lab specification. Rather than generating code, Claude helped me design a modular architecture across seven source files, reviewed it against all lab requirements, and organized a 12-phase roadmap in my development notebook to guide implementation one TODO at a time. No code was written in this session.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration 1

Iteration 1 established the full project scaffold: 8 stub source modules, 6 test files, and 52 tests all failing as expected. During the environment sanity check, the cell raised a hard DeprecatedEnv error for CliffWalking-v0. I asked Claude if we could develop for both v0 and v1. Claude explained that newer Gymnasium versions block v0 entirely and recommended switching to v1. Updated EnvironmentManager and conftest.py accordingly.

Here's how Claude describes what happened:

`Craig came in with a clear philosophy — encapsulation and TDD — and I filled in the details. He named the classes, I proposed the interfaces. He caught the gaps, I fixed them. He added the CLI requirement, I worked out what that meant for the design. The most interesting moment was the reset() method — a small prompt that solved a non-obvious problem in EpisodeRunner before we'd written a line of implementation.`

#### 3.3.2 Iteration 2

Implemented a full vertical slice: TDAgent base class with Q-table initialization, epsilon-greedy action selection, and shared TD update logic. SARSAAgent and QLearningAgent each override _bootstrap — a refactor we made after inspection revealed two identical lines in both subclasses. EnvironmentManager wraps CliffWalking-v1, and EpisodeRunner runs the episode loop across multiple seeds. Validated with 28 passing tests and a behavioral demo showing learning improvement from -515 to -88 mean episode return.

Here's how Claude describes what happened:

`Iteration 2 was where it got interesting. ConstantSchedule went in cleanly, then I violated the TDD workflow by implementing the agents, environment, and runner before confirming the tests failed. Craig caught it immediately. Then during code review, he pushed deeper — asking about alpha, gamma, bootstrapping, the Q-table — and that conversation led to a genuine design improvement. He spotted the duplicated lines in update and asked if we could refactor. The Template Method pattern emerged from that, making the on-policy vs off-policy distinction cleaner than my original implementation. Craig's scrutiny produced better code than I wrote on my own.`


#### 3.3.3 Iteration 3

Implemented LinearDecaySchedule and ExponentialDecaySchedule. Both use an episode counter with epsilon derived on demand via a @property — no stored epsilon value. Linear interpolates between epsilon_start and epsilon_end, clamping at the floor. Exponential applies decay_rate ** episode.

Claude's description:

`Iteration 3 was smoother — the TDD workflow held, red confirmed before a line was written. The most interesting moment came during code review when Craig noticed that value was accessed without parentheses and asked why. That led to a discussion of @property and the design choice to derive epsilon from a counter rather than store it directly. Craig called it elegant, and it is — reset() becomes trivial, and there's no risk of stale state. The one stumble was a stale kernel during the demos, which Craig caught and resolved himself.`

#### 3.4.4 Subsequent Iterations

We iterated a total of 10 cycles, and the pattern was largely the same. During the ninth cycle, I was reviewing the trajectory diagrams and I noticed that the arrows seemed to be indicating a suboptimal, noisy path.  I asked Claude to investigate and we learned the the arrow labels weren't properly mapped.  We fixed the mapping, and the results, accurately reported, aligned to expectations.

Claude's description:




### 3.3 Critical Evaluation

In [ ]:
'''
(50-75 words)

Did you catch any mistakes the AI made?
Did you test alternative approaches?
How did you verify the final solution was correct?
'''

### 3.4 Learning Reflection

In [ ]:
'''
(50-75 words)

What did you learn about the RL algorithm through debugging?
What did you learn about working with AI tools?
What would you do differently next time?
'''

In [ ]:
'''
Grading Criteria

Excellent (30-35 points):
    Documents 3+ specific debugging cycles
    Shows critical evaluation of AI suggestions
    Demonstrates learning from the process
    Specific, detailed examples

Good (24-29 points):
    Documents 2 debugging cycles
    Some critical thinking shown
    Generic learning statements
    Less specific

Adequate (18-23 points):
    Mentions iteration but lacks detail
    Minimal critical evaluation
    Vague descriptions

Poor (0-17 points):
    “I used AI to generate code”
    No iteration documented
    No evidence of critical thinking
    Appears to accept first draft
'''

## Section 4: Speaker Notes

In [ ]:
'''
(~5 minutes / 5-7 bullets)

Weight: 10 points (10%)
Purpose: Practice communicating technical work concisely. These could be used for peer presentations or your own reference.

Outline a brief presentation of your work:

Format: 5-7 bullet points covering:

Problem statement and motivation
Method and key algorithmic choices
Important design decision or challenge you faced
Main result or finding
Key insight or learning
(Optional) Connection to future weeks or real-world applications
Example:

• Problem: Exploration-exploitation in 10-armed bandits with non-stationary rewards
• Method: Implemented ε-greedy and UCB with constant step-size (α=0.1)
• Design choice: Used constant α instead of sample average to track changing rewards
• Key result: UCB outperformed ε-greedy after 500 steps (15% lower regret)
• Insight: Uncertainty-based exploration more efficient than random
• Challenge: Debugging action selection logic took 6 iterations with Claude
• Connection: This exploration concept extends to full MDPs in Week 2
'''